# 01 - Entraînement Multi-Task, 5-Fold et Ensembling avec W&B

**Objectifs de ce notebook :**
1. Créer un Dataset PyTorch qui extrait les deux labels (Marque et Modèle) depuis le nom des fichiers.
2. Définir un modèle ResNet50 avec un tronc commun et deux têtes de classification (MTL).
3. Entraîner 5 versions du modèle via une 5-Fold Cross-Validation stratifiée.
4. Monitorer l'entraînement en temps réel avec Weights & Biases (W&B).
5. Créer la fonction d'extraction finale qui fait la moyenne (Ensembling) des vecteurs des 5 modèles.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.model_selection import StratifiedKFold
import numpy as np
import wandb
from dotenv import load_dotenv

load_dotenv()

wandb.login(key=os.getenv("WANDB_API_KEY"))

DATA_DIR = "../data/raw/Cars"
BATCH_SIZE = 32
EPOCHS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {DEVICE}")

In [ ]:
all_images = [f for f in os.listdir(DATA_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

image_paths = []
brand_labels = []
model_labels = []
composite_labels = []

for img_name in all_images:
    parts = img_name.split('_')
    brand = int(parts[0])
    model = int(parts[1])
    
    image_paths.append(os.path.join(DATA_DIR, img_name))
    brand_labels.append(brand)
    model_labels.append(model)
    composite_labels.append(f"{brand}_{model}")

num_brands = len(set(brand_labels))
num_models = len(set(model_labels))
print(f"Total images : {len(image_paths)} | Marques : {num_brands} | Modèles uniques : {num_models}")

class CarsMTLDataset(Dataset):
    def __init__(self, img_paths, brands, models, transform=None):
        self.img_paths = np.array(img_paths)
        self.brands = np.array(brands)
        self.models = np.array(models)
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        brand = torch.tensor(self.brands[idx], dtype=torch.long)
        model = torch.tensor(self.models[idx], dtype=torch.long)
        
        return image, brand, model

train_transform = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
class MultiTaskResNet(nn.Module):
    def __init__(self, num_brands, num_models):
        super(MultiTaskResNet, self).__init__()
        # Tronc commun
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.shared_backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # Têtes spécifiques
        self.head_brand = nn.Linear(2048, num_brands)
        self.head_model = nn.Linear(2048, num_models)

    def forward(self, x):
        features = self.shared_backbone(x)
        features = torch.flatten(features, 1) # Descripteur de taille 2048
        
        out_brand = self.head_brand(features)
        out_model = self.head_model(features)
        
        return out_brand, out_model, features

In [ ]:
# Initialisation du K-Fold Stratifié
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# On va sauvegarder les chemins vers les meilleurs poids de nos 5 modèles
saved_models_paths = []

# La boucle magique !
for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, composite_labels)):
    print(f"\n{'='*20} Début du FOLD {fold+1}/5 {'='*20}")
    
    # 1. Initialisation de la session W&B pour ce Fold
    run = wandb.init(
        project="cars-retrieval-g05",
        group="MTL-ResNet50-5Fold",
        name=f"Fold_{fold+1}",
        config={"batch_size": BATCH_SIZE, "epochs": EPOCHS, "fold": fold+1}
    )
    
    # 2. Préparation des DataLoaders
    train_dataset = CarsMTLDataset(
        [image_paths[i] for i in train_idx], 
        [brand_labels[i] for i in train_idx], 
        [model_labels[i] for i in train_idx], 
        transform=train_transform
    )
    val_dataset = CarsMTLDataset(
        [image_paths[i] for i in val_idx], 
        [brand_labels[i] for i in val_idx], 
        [model_labels[i] for i in val_idx], 
        transform=val_transform
    )
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 3. Initialisation du Modèle, Loss et Optimiseur
    model = MultiTaskResNet(num_brands, num_models).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    best_val_loss = float('inf')
    model_save_path = f"best_model_fold_{fold+1}.pth"
    
    # 4. Boucle d'entraînement
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        
        for images, brands, models_target in train_loader:
            images, brands, models_target = images.to(DEVICE), brands.to(DEVICE), models_target.to(DEVICE)
            
            optimizer.zero_grad()
            out_brand, out_model, _ = model(images)
            
            # La Loss est la somme des deux tâches !
            loss_brand = criterion(out_brand, brands)
            loss_model = criterion(out_model, models_target)
            loss = loss_brand + loss_model
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # 5. Boucle de validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, brands, models_target in val_loader:
                images, brands, models_target = images.to(DEVICE), brands.to(DEVICE), models_target.to(DEVICE)
                out_brand, out_model, _ = model(images)
                loss = criterion(out_brand, brands) + criterion(out_model, models_target)
                val_loss += loss.item()
                
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        # Envoi des métriques à W&B en direct !
        wandb.log({
            "epoch": epoch+1, 
            "train_loss": avg_train_loss, 
            "val_loss": avg_val_loss
        })
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        # Early Stopping local : Sauvegarde du meilleur modèle du fold
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), model_save_path)
            
    saved_models_paths.append(model_save_path)
    wandb.finish() # Termine la session pour ce fold

In [ ]:
# On crée une liste de nos 5 modèles entraînés
ensemble_models = []
for path in saved_models_paths:
    m = MultiTaskResNet(num_brands, num_models)
    m.load_state_dict(torch.load(path))
    m.to(DEVICE)
    m.eval() # Mode évaluation strict
    ensemble_models.append(m)

print("Les 5 modèles ont été chargés pour l'ensembling !")

def extract_ensemble_descriptor(img_path):
    """
    Prend une image, la passe dans les 5 modèles, 
    et fait la moyenne de leurs vecteurs de 2048 dimensions.
    """
    image = Image.open(img_path).convert('RGB')
    input_tensor = val_transform(image).unsqueeze(0).to(DEVICE) # On utilise val_transform (sans data augmentation)
    
    all_features = []
    with torch.no_grad():
        for m in ensemble_models:
            _, _, features = m(input_tensor) # On ne récupère que le 3ème élément : le tronc commun (2048)
            all_features.append(features.cpu().squeeze())
            
    # Empile les 5 tenseurs et fait la moyenne sur la dimension 0 (les modèles)
    stacked_features = torch.stack(all_features)
    final_descriptor = torch.mean(stacked_features, dim=0)
    
    return final_descriptor.numpy()

# Test sur l'image R1 de votre groupe !
test_img = os.path.join(DATA_DIR, "0_1_BMW_X3_207.jpg")
if os.path.exists(test_img):
    descriptor = extract_ensemble_descriptor(test_img)
    print(f"Taille du descripteur final (Ensemble de 5 modèles) : {descriptor.shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import os


print("Génération d'une mini-base de données de test...")
db_features = np.random.rand(100, 2048) 
db_paths = np.array([f"image_bidon_{i}.jpg" for i in range(100)])

db_features[42] = descriptor 
db_paths[42] = test_img # Ta BMW X3

def search_similar_images(query_desc, db_feats, db_img_paths, top_k=5):
    query_desc_reshaped = query_desc.reshape(1, -1)
    similarities = cosine_similarity(query_desc_reshaped, db_feats)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    best_paths = db_img_paths[top_indices]
    best_scores = similarities[top_indices]
    
    return best_paths, best_scores

print("\nLancement de la recherche...")
TOP_K = 20
results_paths, results_scores = search_similar_images(descriptor, db_features, db_paths, top_k=TOP_K)

# Affichage visuel (Matplotlib)
fig, axes = plt.subplots(1, TOP_K + 1, figsize=(20, 4))

# Afficher la Requête
img_req = Image.open(test_img)
axes[0].imshow(img_req)
axes[0].set_title("IMAGE REQUÊTE\n(BMW X3)", color="blue", fontweight="bold")
axes[0].axis('off')

# Afficher les Résultats Top-K
for i in range(TOP_K):
    path = results_paths[i]
    score = results_scores[i]
    
    # Si le fichier existe vraiment sur ton PC, on l'affiche, sinon on met un carré gris (pour notre test simulé)
    if os.path.exists(path):
        img_res = Image.open(path)
        axes[i+1].imshow(img_res)
    else:
        axes[i+1].imshow(np.zeros((224, 224, 3), dtype=np.uint8) + 200) # Carré gris
        
    # Le score en pourcentage de similarité
    axes[i+1].set_title(f"Top {i+1}\nScore: {score*100:.1f}%\n{os.path.basename(path)[:15]}...")
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()